# Lab 3.5: Iterative Refinement Techniques

**What you'll build:** Three refinement workflows (input/output examples, test-driven iteration, interview pattern) -- and learn why aimless iteration wastes time while structured refinement converges fast.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way -- aimless manual iteration with vague feedback | 5 min |
| 2 | The Right Way -- test-driven iteration that auto-converges | 5 min |
| 3 | Your Turn -- apply the interview pattern + test-driven iteration | 10 min |
| 4 | Stress Test -- compare convergence speed across techniques | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You need a function that transforms user data from a legacy API format to your new internal format. The transformation has specific rules:

| Legacy Field | New Field | Transformation |
|-------------|-----------|----------------|
| `first_name` + `last_name` | `full_name` | Concatenate with space |
| `email_addr` | `email` | Lowercase, strip whitespace |
| `created_at` (ISO string) | `created_ts` | Unix timestamp (integer) |
| `is_active` (string "true"/"false") | `active` | Boolean |
| `role` | `permissions` | Map: "admin" -> ["read","write","delete"], "user" -> ["read"], "editor" -> ["read","write"] |
| `phone` (optional) | `phone` | null if missing, E.164 format if present |

In [ ]:
# Test cases that define the expected behavior
TEST_CASES = [
    {
        "input": {
            "first_name": "Alice",
            "last_name": "Smith",
            "email_addr": "  Alice.Smith@Example.com  ",
            "created_at": "2024-01-15T10:30:00Z",
            "is_active": "true",
            "role": "admin",
            "phone": "+1-555-0123"
        },
        "expected": {
            "full_name": "Alice Smith",
            "email": "alice.smith@example.com",
            "created_ts": 1705314600,
            "active": True,
            "permissions": ["read", "write", "delete"],
            "phone": "+15550123"
        }
    },
    {
        "input": {
            "first_name": "Bob",
            "last_name": "Jones",
            "email_addr": "bob@test.com",
            "created_at": "2023-06-01T00:00:00Z",
            "is_active": "false",
            "role": "user"
        },
        "expected": {
            "full_name": "Bob Jones",
            "email": "bob@test.com",
            "created_ts": 1685577600,
            "active": False,
            "permissions": ["read"],
            "phone": None
        }
    },
    {
        "input": {
            "first_name": "Carol",
            "last_name": "Lee",
            "email_addr": "CAROL@COMPANY.IO",
            "created_at": "2024-12-25T23:59:59Z",
            "is_active": "true",
            "role": "editor",
            "phone": "555-9999"
        },
        "expected": {
            "full_name": "Carol Lee",
            "email": "carol@company.io",
            "created_ts": 1735171199,
            "active": True,
            "permissions": ["read", "write"],
            "phone": "+5559999"
        }
    }
]

print(f"Test cases defined: {len(TEST_CASES)}")
for i, tc in enumerate(TEST_CASES, 1):
    print(f"  Case {i}: {tc['input']['first_name']} {tc['input']['last_name']} "
          f"(role={tc['input']['role']}, phone={'yes' if tc['input'].get('phone') else 'no'})")

---

## Phase 1: The Wrong Approach

Let's try generating the function with a vague prompt and then iterating with aimless feedback.

In [ ]:
# Vague prompt -- what most people start with
VAGUE_PROMPT = """Write a Python function called transform_user that converts user data 
from a legacy format to a new format. The legacy data has fields like first_name, 
last_name, email_addr, created_at, is_active, role, and phone. Transform them to 
the new format with appropriate conversions.

Return ONLY the Python function code, no explanation."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": VAGUE_PROMPT}]
)

vague_code = response.content[0].text.strip()
if vague_code.startswith("```"):
    vague_code = vague_code.split("\n", 1)[1].rsplit("```", 1)[0].strip()

print("=== VAGUE PROMPT OUTPUT ===")
print(vague_code)
print("\n--- Problems we'll likely find: ---")
print("- May not match exact field name mappings")
print("- Role -> permissions mapping probably missing or wrong")
print("- Phone E.164 normalization likely missing")
print("- is_active string -> boolean conversion may be naive")

In [ ]:
# Simulate aimless iteration: vague feedback
VAGUE_FEEDBACK_PROMPT = f"""Here is your function:
```python
{vague_code}
```

It has some issues. The email handling is wrong, the role mapping needs work,
and the phone formatting isn't right. Can you fix these?

Return ONLY the corrected Python function code."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": VAGUE_FEEDBACK_PROMPT}]
)

vague_v2 = response.content[0].text.strip()
if vague_v2.startswith("```"):
    vague_v2 = vague_v2.split("\n", 1)[1].rsplit("```", 1)[0].strip()

print("=== AFTER VAGUE FEEDBACK (iteration 2) ===")
print(vague_v2[:500])
print("\n... Still may not match spec exactly. No way to verify without running tests.")

### Why aimless iteration fails

- **No objective success criterion.** You are manually eyeballing the code each iteration.
- **Vague feedback is ambiguous.** "Email handling is wrong" -- wrong how? The model guesses.
- **Regressions go unnoticed.** Fixing phone formatting might break the role mapping.
- **Human bottleneck.** Every iteration requires your manual review.

---

## Phase 2: The Right Approach

Test-driven iteration: write tests first, then let Claude iterate until tests pass.

In [ ]:
# Test-driven iteration: provide tests as the specification
TDD_PROMPT = f"""Write a Python function called transform_user(legacy_data: dict) -> dict
that transforms user data from a legacy format to a new format.

Here are the exact test cases your function must pass:

{json.dumps(TEST_CASES, indent=2)}

Requirements:
- full_name: concatenate first_name + " " + last_name
- email: lowercase and strip whitespace from email_addr
- created_ts: convert ISO 8601 string to Unix timestamp (integer)
- active: convert string "true"/"false" to Python boolean
- permissions: map role to permission list:
  - "admin" -> ["read", "write", "delete"]
  - "user" -> ["read"]
  - "editor" -> ["read", "write"]
- phone: None if missing, strip non-digit characters except leading + for E.164

Return ONLY the Python function code. Include necessary imports."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": TDD_PROMPT}]
)

tdd_code = response.content[0].text.strip()
if tdd_code.startswith("```"):
    tdd_code = tdd_code.split("\n", 1)[1].rsplit("```", 1)[0].strip()

print("=== TEST-DRIVEN OUTPUT ===")
print(tdd_code)

In [ ]:
# Run the generated code against our tests
import traceback

def run_tests(code_str, test_cases):
    """Execute the generated function and run test cases against it."""
    namespace = {}
    try:
        exec(code_str, namespace)
    except Exception as e:
        return {"error": f"Code execution failed: {e}", "passed": 0, "total": len(test_cases), "failures": []}
    
    if 'transform_user' not in namespace:
        return {"error": "Function 'transform_user' not found", "passed": 0, "total": len(test_cases), "failures": []}
    
    transform_user = namespace['transform_user']
    passed = 0
    failures = []
    
    for i, tc in enumerate(test_cases, 1):
        try:
            result = transform_user(tc['input'])
            expected = tc['expected']
            
            # Compare each field
            field_errors = []
            for key in expected:
                if key not in result:
                    field_errors.append(f"Missing field '{key}'")
                elif result[key] != expected[key]:
                    field_errors.append(f"{key}: got {result[key]!r}, expected {expected[key]!r}")
            
            if not field_errors:
                passed += 1
            else:
                failures.append({"case": i, "errors": field_errors})
        except Exception as e:
            failures.append({"case": i, "errors": [f"Exception: {e}"]})
    
    return {"passed": passed, "total": len(test_cases), "failures": failures}

# Run tests
results = run_tests(tdd_code, TEST_CASES)

print(f"=== TEST RESULTS ===")
print(f"Passed: {results['passed']}/{results['total']}")
if results.get('error'):
    print(f"Error: {results['error']}")
for fail in results.get('failures', []):
    print(f"\n  Case {fail['case']} FAILED:")
    for err in fail['errors']:
        print(f"    - {err}")

if results['passed'] == results['total']:
    print("\nAll tests pass on first attempt with test-driven prompting!")

In [ ]:
# If tests failed, iterate with SPECIFIC error feedback
if results['passed'] < results['total']:
    error_details = json.dumps(results['failures'], indent=2)
    
    RETRY_PROMPT = f"""Your transform_user function has test failures:

```python
{tdd_code}
```

Test failures:
{error_details}

Fix the function so ALL test cases pass. The test cases are:
{json.dumps(TEST_CASES, indent=2)}

Return ONLY the corrected Python function code with imports."""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=1000,
        messages=[{"role": "user", "content": RETRY_PROMPT}]
    )
    
    tdd_code_v2 = response.content[0].text.strip()
    if tdd_code_v2.startswith("```"):
        tdd_code_v2 = tdd_code_v2.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    
    results_v2 = run_tests(tdd_code_v2, TEST_CASES)
    print(f"=== RETRY RESULTS ===")
    print(f"Passed: {results_v2['passed']}/{results_v2['total']}")
    for fail in results_v2.get('failures', []):
        print(f"  Case {fail['case']}: {fail['errors']}")
    
    if results_v2['passed'] == results_v2['total']:
        print("All tests pass after one retry with specific error feedback!")
else:
    print("Tests already passed -- no retry needed.")

---

## Phase 3: Your Turn

Now apply the **interview pattern** combined with **test-driven iteration**.

Task: Build a function that formats order summaries for display. But instead of giving all requirements upfront, use the interview pattern to elicit them.

In [ ]:
# =============================================================
# TODO: Use the interview pattern to gather requirements
# =============================================================

# Step 1: Ask Claude to ask clarifying questions
INTERVIEW_PROMPT = """I need a Python function called format_order_summary that takes
an order dict and returns a formatted string for display to the user.

Before writing any code, ask me 3-5 clarifying questions about:
- What fields does the order dict contain?
- What format should the output string be in?
- How should edge cases be handled?
- Any specific formatting requirements (currency, dates, etc.)?

Ask the questions as a numbered list. Do not write any code yet."""

response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    messages=[{"role": "user", "content": INTERVIEW_PROMPT}]
)

questions = response.content[0].text.strip()
print("=== Claude's Questions ===")
print(questions)

In [ ]:
# Step 2: Answer the questions with specific requirements
# TODO: Provide your answers, then ask Claude to generate the function

YOUR_ANSWERS = """
# TODO: Answer Claude's questions with specific requirements
# For example:
# 1. The order dict has: order_id (str), items (list of dicts with name, qty, price),
#    subtotal (float), tax (float), total (float), created_at (ISO string)
# 2. Output format: multi-line string with header, item list, and total section
# 3. Handle: empty items list (show "No items"), missing optional fields
# 4. Currency: USD format ($X,XXX.XX), dates: "Jan 15, 2024" format
"""

GENERATE_PROMPT = f"""Here are my answers to your questions:

{YOUR_ANSWERS}

Now write the format_order_summary function. Return ONLY the Python function code."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": INTERVIEW_PROMPT},
        {"role": "assistant", "content": questions},
        {"role": "user", "content": GENERATE_PROMPT}
    ]
)

interview_code = response.content[0].text.strip()
if interview_code.startswith("```"):
    interview_code = interview_code.split("\n", 1)[1].rsplit("```", 1)[0].strip()

print("=== Generated after interview ===")
print(interview_code)

In [ ]:
# =============================================================
# Checker: validates your interview pattern usage
# =============================================================

def check_interview_pattern(answers, generated_code):
    errors = []
    
    # Check 1: Answers should be specific, not vague
    if 'TODO' in answers and len(answers.strip().split('\n')) < 5:
        errors.append("Answers still contain TODO -- provide specific requirements.")
    
    # Check 2: Generated code should exist
    if not generated_code or len(generated_code) < 50:
        errors.append("No meaningful code generated.")
    
    # Check 3: Generated code should reference format_order_summary
    if 'format_order_summary' not in generated_code:
        errors.append("Generated code does not define format_order_summary function.")
    
    print("=== INTERVIEW PATTERN VALIDATION ===")
    if not errors:
        print("PASSED -- Interview pattern produced a targeted implementation.")
        print("The function was generated with full context from your answers.")
    else:
        for e in errors:
            print(f"  [ERROR] {e}")
    
    return len(errors) == 0

check_interview_pattern(YOUR_ANSWERS, interview_code)

---

## Phase 4: Stress Test

Compare convergence speed: how many iterations does each technique need?

In [ ]:
# Compare: vague prompting vs test-driven on the original transform_user task
print("=== CONVERGENCE COMPARISON ===")
print()

# Technique 1: Vague prompt (already done in Phase 1)
print("Technique 1: Vague Prompt")
vague_results = run_tests(vague_code, TEST_CASES)
vague_score = "{}/{}".format(vague_results["passed"], vague_results["total"])
print(f"  Iteration 1: {vague_score} tests pass")
print(f"  Would need multiple manual review cycles to fix remaining issues")

print()

# Technique 2: Test-driven (already done in Phase 2)
print("Technique 2: Test-Driven Iteration")
tdd_results = run_tests(tdd_code, TEST_CASES)
tdd_score = "{}/{}".format(tdd_results["passed"], tdd_results["total"])
print(f"  Iteration 1: {tdd_score} tests pass")
if tdd_results['passed'] == tdd_results['total']:
    print(f"  Converged in 1 iteration!")
else:
    print(f"  Would auto-retry with specific errors (no manual review needed)")

print()
print("Technique 3: Input/Output Examples")
# Show the first test case as an example
IO_PROMPT = f"""Write a Python function called transform_user(legacy_data: dict) -> dict.

Here is an example of the transformation:

INPUT:
{json.dumps(TEST_CASES[0]['input'], indent=2)}

OUTPUT:
{json.dumps(TEST_CASES[0]['expected'], indent=2)}

Write the function that performs this transformation for any valid input.
Return ONLY the Python function code with imports."""

response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": IO_PROMPT}]
)
io_code = response.content[0].text.strip()
if io_code.startswith("```"):
    io_code = io_code.split("\n", 1)[1].rsplit("```", 1)[0].strip()

io_results = run_tests(io_code, TEST_CASES)
io_score = "{}/{}".format(io_results["passed"], io_results["total"])
print(f"  Iteration 1 (from 1 example): {io_score} tests pass")

print()
print("=== SUMMARY ===")
print("{:<25} {:>12} {:>15}".format("Technique", "Pass Rate", "Manual Review"))
print("{} {} {}".format("-" * 25, "-" * 12, "-" * 15))
print("{:<25} {:>12} {:>15}".format("Vague prompt", vague_score, "Every iteration"))
print("{:<25} {:>12} {:>15}".format("Input/output example", io_score, "Every iteration"))
print("{:<25} {:>12} {:>15}".format("Test-driven", tdd_score, "None (auto)"))

### Key Takeaways

1. **Test-driven iteration** is the most reliable technique because tests provide an objective, automated success criterion.
2. **Input/output examples** teach transformation patterns effectively but do not catch regressions.
3. **The interview pattern** front-loads context gathering, producing better first drafts and fewer iterations.
4. **Vague feedback** ("fix the email handling") is ambiguous. Specific error messages from tests are unambiguous.
5. **Combine techniques:** Interview to gather requirements, then test-driven iteration to verify correctness.